In [7]:
import os
import sys

# 현재 작업 디렉토리 기준으로 상위 1단계 폴더를 루트로 설정
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# print("프로젝트 루트로 설정된 경로:", project_root)

In [8]:
# 데이터 확인하기 2025.11.21
# 이상인 컬럼 제거 후 RandomForest 기본 모델 돌리기
import pandas as pd
import numpy  as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler # 데이터 전처리용
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials

import matplotlib.pyplot as plt
import seaborn as sns

import importlib
from utils import preprocessing

# 모듈 reload
importlib.reload(preprocessing)
# importlib.reload(user_utils)

from utils.preprocessing import load_data, split_features_target, scale_data, data_split, remove_zero_columns
from utils.user_utils    import get_model_train_eval

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

# train = pd.read_csv("../data/train.csv")
# test  = pd.read_csv("../data/test.csv")

c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\hyperopt\atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [9]:
# 데이터 로딩
train, test = load_data()

In [10]:
# 데이터 할당
X_features, y_labels = split_features_target(train) # ID와 TARGET 모두 제거하고 X_train 만들기
X_test     = test.drop(columns=['ID'], axis=1) # test 데이터에서도 ID 제거
X_features['var3'] = X_features['var3'].replace(-999999, 2)
# var3 의 최소값 -99999 를 최빈값으로 변경하기

In [16]:
# df.info()

# print("\n 결측값의 수:", df.isna().sum().sum())

# <class 'pandas.core.frame.DataFrame'>
# RangeIndex: 76020 entries, 0 to 76019
# Columns: 369 entries, var3 to var38
# dtypes: float64(111), int64(258)
# memory usage: 214.0 MB

#  결측값의 수: 0

In [11]:
# 학습/테스트 데이터 분리
X_train, X_val, y_train, y_val = data_split(
  X_features,
  y_labels,
)

In [12]:
# 2) Scaler 생성 (train에만 fit)
scaler = StandardScaler()
scaler.fit(X_train)

,copy,True
,with_mean,True
,with_std,True


In [13]:
# 3) train, val, test에 동일한 scaler 적용
X_train_scaled = scaler.transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

In [14]:
# 레이블의 분포 확인
cust_cnt = y_labels.value_counts()
print(cust_cnt) # 1이 불만족 3008명, 만족이 73012

# 불만족고객의 비율
cust_rate = cust_cnt[1] / cust_cnt.sum()
print(f'불만족 고객 비율: {cust_rate:.2f}')

TARGET
0    73012
1     3008
Name: count, dtype: int64
불만족 고객 비율: 0.04


In [15]:
class ThresholdModel:
    def __init__(self, base_model, threshold):
        self.base_model = base_model
        self.threshold = threshold

    def fit(self, X, y):
        # 재학습 방지 — 이미 fit된 모델 그대로 사용
        return self

    def predict(self, X):
        proba = self.base_model.predict_proba(X)[:, 1]
        return (proba >= self.threshold).astype(int)

    def predict_proba(self, X):
        return self.base_model.predict_proba(X)

In [16]:
# ===========================
# 1) HyperOpt objective 함수
# ===========================
def objective(params):
    # 정수형 변환
    params['max_depth'] = int(params['max_depth'])
    params['n_estimators'] = int(params['n_estimators'])

    model = XGBClassifier(
        **params,
        eval_metric='logloss',
        random_state=42
    )

    model.fit(X_train_scaled, y_train)
    proba = model.predict_proba(X_val_scaled)[:, 1]
    pred = (proba >= 0.5).astype(int)
    f1 = f1_score(y_val, pred)

    return {'loss': -f1, 'status': STATUS_OK}

In [21]:
# ===========================
# 2) 검색 공간 정의
# ===========================
search_space = {
    'n_estimators': hp.quniform('n_estimators', 200, 800, 50),
    'learning_rate': hp.loguniform('learning_rate', -4, -2),    # 0.018 ~ 0.135
    'max_depth': hp.quniform('max_depth', 3, 8, 1),
    'subsample': hp.uniform('subsample', 0.6, 0.9),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.6, 0.9),
}

In [22]:
# ===========================
# 3) HyperOpt 실행
# ===========================
trials = Trials()
best_params = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=30,   # 30번 정도면 충분
    trials=trials,
    rstate=np.random.default_rng(42)
)

# 정수형 변환
best_params['n_estimators'] = int(best_params['n_estimators'])
best_params['max_depth'] = int(best_params['max_depth'])

100%|██████████| 30/30 [02:27<00:00,  4.91s/trial, best loss: -0.03536977491961415] 


In [23]:
# ===========================
# 4) HyperOpt로 찾은 파라미터로 모델 생성
# ===========================
xgb_hopt = XGBClassifier(
    **best_params,
    eval_metric='logloss',
    random_state=42
)

In [24]:
xgb_hopt.fit(X_train_scaled, y_train)
proba_hopt = xgb_hopt.predict_proba(X_val_scaled)[:,1]

get_model_train_eval(xgb_hopt, "XGB_100_HP",
    X_train, X_val,
    y_train, y_val
)

# threshold 탐색
best_f1 = 0
best_threshold = 0
thresholds = np.arange(0.01, 0.50, 0.01)

for thr in thresholds:
    pred_thr = (proba_hopt >= thr).astype(int)
    f1 = f1_score(y_val, pred_thr)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = thr

threshold_hopt = ThresholdModel(xgb_hopt, best_threshold)

get_model_train_eval(
    threshold_hopt,
    f'XGB_100_HP_thr_{best_threshold:.2f}',
    X_train, X_val,
    y_train, y_val
)

✓ 모델 저장 완료: models\XGB_100_HP.pkl
  파일 크기: 0.53 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8428, 정확도: 0.9605, 정밀도: 0.5500, 재현율: 0.0183, F1: 0.0354
오차행렬:
[[14593     9]
 [  591    11]]
실행 시간: 2.925999164581299
✓ 모델 저장 완료: models\XGB_100_HP_thr_0.15.pkl
  파일 크기: 0.53 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8428, 정확도: 0.9146, 정밀도: 0.2189, 재현율: 0.4502, F1: 0.2946
오차행렬:
[[13635   967]
 [  331   271]]
실행 시간: 0.10498809814453125


In [28]:
class ThresholdModel_rf:
    def __init__(self, base_model, threshold):
        self.base_model = base_model
        self.threshold = threshold

    def fit(self, X, y):
        # fit을 무시하고, 이미 fit된 모델 그대로 사용
        return self

    def predict(self, X):
        proba = self.base_model.predict_proba(X)[:, 1]
        return (proba >= self.threshold).astype(int)

    def predict_proba(self, X):
        return self.base_model.predict_proba(X)

In [29]:
# ================================
# 1) class_weight 옵션 정의
# ================================
class_weight_options = [
    None,
    {0:1, 1:2},
    {0:1, 1:3},
    {0:1, 1:4}
]

In [30]:
# ================================
# 2) HyperOpt Objective 함수
# ================================
def objective_rf(params):

    # 정수 변환
    params['n_estimators'] = int(params['n_estimators'])
    params['max_depth'] = int(params['max_depth'])
    params['min_samples_split'] = int(params['min_samples_split'])
    params['min_samples_leaf'] = int(params['min_samples_leaf'])

    # class_weight idx → 실제 값
    cw_idx = int(params['class_weight_idx'])
    params['class_weight'] = class_weight_options[cw_idx]
    del params['class_weight_idx']

    model = RandomForestClassifier(
        **params,
        n_jobs=-1,
        random_state=0
    )

    # 랜포는 스케일 없이도 OK
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_val)[:, 1]
    pred = (proba >= 0.5).astype(int)
    f1 = f1_score(y_val, pred)

    return {'loss': -f1, 'status': STATUS_OK}

In [31]:
# ================================
# 3) HyperOpt Search Space 설정
# ================================
search_space_rf = {
    'n_estimators':     hp.quniform('n_estimators', 100, 500, 50),
    'max_depth':        hp.quniform('max_depth', 6, 20, 1),
    'min_samples_split': hp.quniform('min_samples_split', 2, 15, 1),
    'min_samples_leaf':  hp.quniform('min_samples_leaf', 1, 8, 1),
    'max_features':     hp.choice('max_features', ['sqrt', None]),
    'class_weight_idx': hp.choice('class_weight_idx', [0, 1, 2, 3])
}

# ================================
# 4) HyperOpt 실행
# ================================
trials = Trials()
best_rf_params = fmin(
    fn=objective_rf,
    space=search_space_rf,
    algo=tpe.suggest,
    max_evals=25,        # RF는 느리므로 25회 추천
    trials=trials,
    rstate=np.random.default_rng(42)
)

# 정수 변환 + class_weight 매핑
best_rf_params['n_estimators'] = int(best_rf_params['n_estimators'])
best_rf_params['max_depth'] = int(best_rf_params['max_depth'])
best_rf_params['min_samples_split'] = int(best_rf_params['min_samples_split'])
best_rf_params['min_samples_leaf'] = int(best_rf_params['min_samples_leaf'])

best_rf_params['class_weight'] = class_weight_options[int(best_rf_params['class_weight_idx'])]
del best_rf_params['class_weight_idx']

print("\n===== RF HyperOpt Best Params =====")
print(best_rf_params)

  0%|          | 0/25 [00:00<?, ?trial/s, best loss=?]

 20%|██        | 5/25 [01:34<06:16, 18.80s/trial, best loss: -0.006611570247933884]


KeyboardInterrupt: 

In [27]:
# -----------------------------
# 6. RandomForest
# -----------------------------

rf = RandomForestClassifier(
    **best_rf_params,
    n_jobs=-1,
    random_state=0
)

rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_val)[:, 1]

get_model_train_eval(rf,'RF_100_HP_max24_est150_classWeight2',
    X_train, X_val,
    y_train, y_val
)

best_f1 = 0
best_threshold = 0
thresholds = np.arange(0.01, 0.50, 0.01)

for thr in thresholds:
    pred_thr = (rf_proba >= thr).astype(int)   # 해당 코드는 proba_val 부분만 바꿔서 재활용
    f1 = f1_score(y_val, pred_thr)              # 그 외에 변수 재선언 할 필요 없다
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = thr

print(f"\nBest Threshold: {best_threshold:.2f}, Best F1: {best_f1:.4f}")

# 최적 threshold로 성능 출력
# pred_best = (rf_proba >= best_threshold).astype(int)
# get_clf_eval(y_val, pred_best, rf_proba)
threshold_model_rf = ThresholdModel_rf(rf, best_threshold)

get_model_train_eval(threshold_model_rf, f"RF_100_HP_max24_est150_classWeight2_thr_{best_threshold:.2f}",
    X_train, X_val,
    y_train, y_val
)

InvalidParameterError: The 'class_weight' parameter of RandomForestClassifier must be a str among {'balanced_subsample', 'balanced'}, an instance of 'dict', an instance of 'list' or None. Got np.int64(2) instead.

In [23]:
# ============================================
# 6. TEST.CSV에 대한 최종 예측
#
# TARGET=1(불만족)을 얼마나 잘 잡아내는지
# 즉, “잠재적으로 문제가 생길 고객”을 잘 찾아내는지?
#
# 불만족(=1) 고객 비율이 너무 낮기 때문에
# F1 score과 recall이 낮게 나오는 것이 정상이고,
# AUC를 중심으로 보는 게 맞아.
# ============================================

# XGBoost test 예측값
xgb_test_pred = xgb.predict_proba(X_test_scaled)[:, 1]

# RandomForest test 예측값
rf_test_pred = rf.predict_proba(X_test_only)[:, 1]

print("\n===== FINAL TEST PREDICTIONS =====")
print("\nXGBoost Test Predictions (probability of TARGET=1(불만족)):")
print(xgb_test_pred[:20])   # 상위 20개만 미리보기

print("\nRandomForest Test Predictions (probability of TARGET=1(불만족)):")
print(rf_test_pred[:20])    # 상위 20개만 미리보기


===== FINAL TEST PREDICTIONS =====

XGBoost Test Predictions (probability of TARGET=1(불만족)):
[0.01531759 0.01531759 0.01483977 0.01531759 0.01531759 0.01531759
 0.02546733 0.01531759 0.01439818 0.01546898 0.01531759 0.01531759
 0.01786462 0.01531759 0.01054986 0.01547328 0.01483977 0.01531759
 0.01531759 0.01776981]

RandomForest Test Predictions (probability of TARGET=1(불만족)):
[0.06008547 0.06054385 0.0215892  0.07928155 0.02017998 0.24645532
 0.06404603 0.23378981 0.03953955 0.04020289 0.04272866 0.02776394
 0.03454718 0.02182546 0.03519108 0.05251217 0.12909037 0.02113416
 0.02213186 0.03631033]


In [24]:
best_f1 = 0
best_threshold = 0
thresholds = np.arange(0.01, 0.50, 0.01)

for thr in thresholds:
    pred_thr = (proba_val >= thr).astype(int)   # 해당 코드는 proba_val 부분만 바꿔서 재활용
    f1 = f1_score(y_val, pred_thr)              # 그 외에 변수 재선언 할 필요 없다
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = thr

print(f"Best Threshold: {best_threshold:.2f}, Best F1: {best_f1:.4f}")

# 최적 threshold로 성능 출력
pred_best = (proba_val >= best_threshold).astype(int)
get_clf_eval(y_val, pred_best, proba_val)

# -----------------------------
# RandomForest Threshold Optimization for F1
# -----------------------------



print(f"[RF] Best Threshold: {best_threshold:.2f}, Best F1 Score: {best_f1:.4f}")

# 최적 threshold로 평가 지표 출력
rf_pred_best = (rf_proba >= best_threshold).astype(int)
get_clf_eval(y_val, rf_pred_best, rf_proba, model_name='RF_best')


Best Threshold: 0.13, Best F1: 0.2974
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8509, 정확도: 0.9036, 정밀도: 0.2090, 재현율: 0.5150, F1: 0.2974
오차행렬:
[[13429  1173]
 [  292   310]]
[RF] Best Threshold: 0.13, Best F1 Score: 0.2974
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8264, 정확도: 0.8630, 정밀도: 0.1675, 재현율: 0.6196, F1: 0.2637
오차행렬:
[[12748  1854]
 [  229   373]]
